## Objectives
- How to collect data?
- How to handle certain values before processing Missing values?
- How to manage memory usage?


In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

In [ ]:
# Load the Knee Replacement CCG data
file_path = f"../merged/Knee Replacement Provider.csv"
df = pd.read_csv(file_path, sep=';')

# Make a copy of the original dataframe
df_original = df.copy()
print("Original dataframe copied to df_original.")

C:\Users\mittall\AppData\Local\Temp\ipykernel_19508\3627958677.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=';')


Original dataframe copied to df_original.


In [3]:
# inspect memory usage: orignal is 85.3 MB
df.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124844 entries, 0 to 124843
Columns: 82 entries, Provider Code to CSVYear
dtypes: float64(5), int64(69), object(8)
memory usage: 78.1+ MB


In [13]:
# Replace '*' suppression markers (used in NHS data for small counts) with NaN
df.replace('*', np.nan, inplace=True)

# reduce memory: strings to categories
# see: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html
# see: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')
    
# int64 to smallest, unsigned int
for col in df.select_dtypes(include='int64').columns:
    df[col] = pd.to_numeric(df[col], downcast='unsigned')
    
# float64 to smallest float
for col in df.select_dtypes(include='float64').columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

# Category columns with all-numeric string categories (e.g. Gender: '0','1','2') cause
# pyarrow serialisation errors. Convert those to float32 so parquet can handle them.
for col in df.select_dtypes(include='category').columns:
    try:
        df[col] = pd.to_numeric(df[col], errors='raise').astype('float32')
    except (ValueError, TypeError):
        pass  # keep non-numeric categories as-is

# memory footprint now less than a fifth: 16 MB
df.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124844 entries, 0 to 124843
Columns: 82 entries, Provider Code to CSVYear
dtypes: category(7), float32(6), uint16(2), uint32(2), uint8(65)
memory usage: 23.5 MB


In [16]:

df.to_parquet(f'./data/interim/1.1-Hip-Reduced.parquet', compression='gzip')